<a href="https://colab.research.google.com/github/Nirzaree/STAC-spec/blob/stac-spec-common/notebooks/generate_stac_pan_india_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:

%pip install --quiet geopandas fiona

%pip install rasterio

%pip install pystac



In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os

import json
import xml.etree.ElementTree as ET
import datetime
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon
from IPython.display import Image, display

import numpy as np
import pystac
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


In [2]:

#vector_path = "/content/drive/MyDrive/stactest/Aquifer_vector.geojson" #101 MB
#vector_path = "/content/drive/MyDrive/stactest/State_pan_india.geojson"
#vector_path='/content/drive/MyDrive/stactest/Watershed_pan_india.geojson' ##459 MB

#vector_path='/content/drive/MyDrive/Copyofpan_india_drainage_lines.geojson' ##12 GB
#vector_path='/content/drive/MyDrive/CopyofMicrowatershed_boundries_v1.geojson'##6.35GB
vector_path='/content/drive/MyDrive/CopyofFirst_census_of_water_bodies.geojson' ## 3GB


In [3]:
gdf = gpd.read_file(vector_path)

In [4]:
print("CRS:", gdf.crs)
print("Number of features:", len(gdf))
print("Bounds (minx, miny, maxx, maxy):", gdf.total_bounds)
print("Geometry types:", gdf.geometry.type.value_counts().to_dict())
print("Columns / properties:", list(gdf.columns))


print(gdf.dtypes)

CRS: EPSG:4326
Number of features: 1760724
Bounds (minx, miny, maxx, maxy): [nan nan nan nan]
Geometry types: {'MultiPoint': 1760724}
Columns / properties: ['id', 'Block/Tehsil Name', 'District Name', 'State Name', 'Village Name', 'area_encroached_percentage', 'basin_name', 'cca_water_body', 'construcion_year', 'construction_cost', 'estimated_cost', 'extent_of_area_covered_by_wua', 'filled_up_storage_name', 'filled_up_storage_space_name', 'ipc_water_body', 'irrigation_potential_revived', 'ismissing', 'khasra_number', 'latitude_dec', 'layer', 'longitude_dec', 'manmade_water_body_type_name', 'max_depth_water_body_fully_filled', 'nature_of_storage', 'no_people_benefited_by_water_body', 'no_town_cities_benefited', 'no_villages_benefited', 'number_of_wua_formed', 'path', 'reason_water_body_in_use_name2', 'reason_water_body_in_use_name3', 'ref_reason_water_body_in_use_id1_name', 'ref_selection_id_dip_sip_exists_name', 'ref_selection_id_encroachment_assessed_name', 'ref_selection_id_water_bod

In [5]:
def read_vector_data(vector_path,
                     target_crs='4326'
                     ):
    vector_gdf = gpd.read_file(vector_path)
    vector_gdf = vector_gdf.to_crs(epsg=target_crs)
    bounds = vector_gdf.total_bounds
    bbox = [float(b) for b in bounds]
    geom = mapping(vector_gdf.union_all())

    id = os.path.basename(vector_path)

    return (vector_gdf,bounds,bbox,geom,id)

In [6]:
vector_gdf,bounds,bbox,geom,id = read_vector_data(vector_path=vector_path)

In [7]:
geom

{'type': 'GeometryCollection', 'geometries': []}

In [8]:
bbox

[nan, nan, nan, nan]

In [9]:
#vector_gdf['Principal_'] = vector_gdf['Principal_'].replace('', np.nan) ## For aquifer layer


In [10]:
item_id='Waterbody census'

vector_item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bbox,
    datetime=datetime.datetime.now(datetime.timezone.utc),
    properties={
        # "title": title,
        # "description": f"Vector data for {os.path.splitext(vector_filename)[0]} in {block} of {state_title}",
        # "start_datetime": start_date.isoformat() + 'Z',
        # "end_datetime": end_date.isoformat() + 'Z',
    }
)


In [11]:
data_dir = "../data/"
corestack_dir = os.path.join(data_dir, 'CorestackCatalogs')

output_dir = os.path.join(data_dir,'STAC_output')

In [12]:
vector_thumbnail_path = os.path.join(
    output_dir,
    vector_path.split('.')[0] + '_thumbnail.png'
)
vector_thumbnail_path

'/content/drive/MyDrive/CopyofFirst_census_of_water_bodies_thumbnail.png'

In [13]:
def rgba_to_hex(rgba_tuple):
    if rgba_tuple is None:
        return '#808080'  # Default gray
    r, g, b, a = rgba_tuple
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

def extract_styling_info(symbol_element):
    fill_color = None
    outline_color = None
    line_width = None

    if symbol_element is None:
        return fill_color, outline_color, line_width


    fill_layer = symbol_element.find('.//layer[@class="SimpleFill"]')
    if fill_layer is not None:
        color_option = fill_layer.find('Option[@name="color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                fill_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                fill_color = None

        outline_option = fill_layer.find('Option[@name="outline_color"]')
        if outline_option is not None:
            try:
                rgb_parts = [int(p) for p in outline_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None

        width_option = fill_layer.find('Option[@name="outline_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None


    line_layer = symbol_element.find('.//layer[@class="SimpleLine"]')
    if line_layer is not None:
        color_option = line_layer.find('Option[@name="line_color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None

        width_option = line_layer.find('Option[@name="line_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None

    return fill_color, outline_color, line_width

In [14]:
def parse_qml_style(qml_url):

    try:
        response = requests.get(qml_url)
        response.raise_for_status()


        qml_in_memory = BytesIO(response.content)

        tree = ET.parse(qml_in_memory)
        root = tree.getroot()
        renderer_element = root.find('.//renderer-v2')

        if renderer_element is None:
            print("No renderer-v2 element found.")
            return None

        renderer_type = renderer_element.get('type')
        style = {'renderer_type': renderer_type}
        symbols = {s.get('name'): s for s in root.findall('.//symbols/symbol')}

        if renderer_type == 'singleSymbol':
            symbol_element = renderer_element.find('.//symbol') or symbols.get(renderer_element.get('symbol'))
            if symbol_element is not None:
                color_option = symbol_element.find('.//layer/Option[@name="line_color"]') or symbol_element.find('.//layer/Option[@name="color"]')
                if color_option is not None:
                    color_value = color_option.get('value').split(',')[0:3]
                    rgb_parts = [int(p) for p in color_value]
                    style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                else:
                    color_prop = symbol_element.find('.//prop[@k="color"]')
                    if color_prop is not None:
                        rgb_parts = [int(p) for p in color_prop.get('v').split(',')[:3]]
                        style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                    else:
                        print(f"Warning: Single symbol color not found in {qml_url}.")
                        return None
            else:
                print(f"Warning: Could not find symbol element for singleSymbol in {qml_url}.")
                return None

        elif renderer_type == 'categorizedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['categories'] = []
            for cat in renderer_element.findall('categories/category'):
                symbol_element = cat.find('symbol') or symbols.get(cat.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['categories'].append({
                    'value': cat.get('value'),
                    'label': cat.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        elif renderer_type == 'graduatedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['classes'] = []
            for cls in renderer_element.findall('classes/class'):
                symbol_element = cls.find('symbol') or symbols.get(cls.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['classes'].append({
                    'lower_bound': float(cls.get('lower')),
                    'upper_bound': float(cls.get('upper')),
                    'label': cls.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        elif renderer_type == 'RuleRenderer':
            style['rules'] = []
            for rule in renderer_element.findall('.//rule'):
                symbol_element = rule.find('.//symbol')
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['rules'].append({
                    'filter': rule.get('filter'),
                    'label': rule.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        else:
            print(f"Warning: Unsupported renderer type '{renderer_type}'. Using default style.")
            return None

        return style
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the QML file from URL: {e}")
        return None
    except ET.ParseError as e:
        print(f"Error parsing XML from QML file at {qml_url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while parsing QML file from {qml_url}: {e}")
        return None

if __name__ == "__main__":

    #qml_url = 'https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Hydrology/Aquifer_style.qml'
    qml_url='https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Demographic/Administrative-Boundary-Style.qml'

    print(f"Parsing QML file from URL: {qml_url}")
    style_info = parse_qml_style(qml_url)

    if style_info:
        print("\nSuccessfully parsed QML style:")
        print(style_info)
    else:
        print("\nFailed to parse QML style.")





Parsing QML file from URL: https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Demographic/Administrative-Boundary-Style.qml

Failed to parse QML style.


/tmp/ipython-input-699924351.py:23: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  symbol_element = renderer_element.find('.//symbol') or symbols.get(renderer_element.get('symbol'))


In [15]:
def generate_vector_thumbnail(vector_gdf,
                              qml_url,
                              out_path
                              ):

    try:
        vector_gdf = gpd.read_file(vector_path)
        style_info = parse_qml_style(qml_url)

        fig, ax = plt.subplots(figsize=(6, 6))

        default_fill_color = (0.8, 0.8, 0.8, 1.0) # Light gray
        default_outline_color = (0, 0, 0, 1.0)   # Black
        default_line_width = 1.0

        if style_info is None:
            print("Applying default style due to parsing error.")
            vector_gdf.plot(ax=ax,
                            color=rgba_to_hex(default_fill_color),
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)

        elif style_info.get('renderer_type') == 'singleSymbol':
            print("Applying single symbol style...")
            fill_color = style_info.get('fill_color', default_fill_color)
            outline_color = style_info.get('outline_color', default_outline_color)
            line_width = style_info.get('line_width', default_line_width)
            vector_gdf.plot(ax=ax,
                            color=rgba_to_hex(fill_color),
                            edgecolor=rgba_to_hex(outline_color),
                            linewidth=line_width)

        elif style_info.get('renderer_type') == 'categorizedSymbol':
            print("Applying categorized style...")

            color_map = {
                cat.get('value'): rgba_to_hex(cat.get('fill_color', default_fill_color))
                for cat in style_info.get('categories', [])
            }

            outline_color_map = {
                cat.get('value'): rgba_to_hex(cat.get('outline_color', default_outline_color))
                for cat in style_info.get('categories', [])
            }

            attribute_name = style_info.get('attribute')

            if attribute_name not in vector_gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                vector_gdf.plot(ax=ax,
                                color=rgba_to_hex(default_fill_color),
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)
            else:
                vector_gdf['mapped_value'] = vector_gdf[attribute_name].apply(lambda x: str(x).strip() if pd.notnull(x) else None)

                fill_colors = vector_gdf['mapped_value'].map(color_map)
                fill_colors = fill_colors.fillna(rgba_to_hex(default_fill_color))

                outline_colors = vector_gdf['mapped_value'].map(outline_color_map)
                outline_colors = outline_colors.fillna(rgba_to_hex(default_outline_color))

                vector_gdf.plot(ax=ax, color=fill_colors, edgecolor=outline_colors, linewidth=default_line_width)


        elif style_info.get('renderer_type') == 'graduatedSymbol':
            print("Applying graduated style...")
            attribute_name = style_info.get('attribute')
            if attribute_name not in vector_gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                vector_gdf.plot(ax=ax,
                                color=rgba_to_hex(default_fill_color),
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)
            else:
                fill_colors = []
                for _, row in vector_gdf.iterrows():
                    val = row[attribute_name]
                    found_color = default_fill_color
                    for cls in style_info.get('classes', []):
                        if cls.get('lower_bound') is not None and cls.get('upper_bound') is not None:
                            if cls['lower_bound'] <= val < cls['upper_bound']:
                                found_color = cls.get('fill_color', default_fill_color)
                                break
                    fill_colors.append(rgba_to_hex(found_color))

                vector_gdf.plot(ax=ax,
                                color=fill_colors,
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)

        elif style_info.get('renderer_type') == 'RuleRenderer':
            print("Applying rule-based style...")
            fill_colors = []
            for _, row in vector_gdf.iterrows():
                assigned_color = default_fill_color
                for rule in style_info.get('rules', []):
                    try:
                        attribute_name = rule['filter'].split(' ')[0].strip().strip('"').strip("'")
                        if attribute_name in row and pd.eval(rule['filter'], local_dict={attribute_name: row[attribute_name]}):
                            assigned_color = rule.get('fill_color', default_fill_color)
                            break
                    except Exception:
                        continue
                fill_colors.append(rgba_to_hex(assigned_color))

            vector_gdf.plot(ax=ax,
                            color=fill_colors,
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)

        else:
            print("Applying default blue style.")
            vector_gdf.plot(ax=ax,
                            color='lightblue',
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)

        ax.set_axis_off()
        plt.tight_layout()
        plt.savefig(out_path)
        plt.close(fig)
        print(f"Thumbnail saved to: {out_path}")

    except Exception as e:
        print(f"Error generating vector thumbnail: {e}")

In [ ]:
generate_vector_thumbnail(vector_gdf=vector_gdf,
                              qml_url=qml_url,
                              out_path=vector_thumbnail_path)

In [ ]:
display(Image(filename=vector_thumbnail_path))

In [ ]:
vector_item

### **Used ijson python package for reading the large GEOJSON files and get the metadata **

In [3]:
pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.3/148.3 kB 5.9 MB/s eta 0:00:00


In [9]:
import ijson

geojson_filepath = '/content/drive/MyDrive/Copyofpan_india_drainage_lines.geojson'

def get_first_feature_geometry(filepath):
        with open(filepath, 'r') as f:
            for item in ijson.items(f, 'features.item.geometry', use_float=True):
                geometry = item
                break
        return geometry


In [10]:
feature_geometry = get_first_feature_geometry(geojson_filepath)
print(feature_geometry)

{'type': 'LineString', 'coordinates': [[93.1808314161215, 22.41777827721593], [93.18111233948785, 22.41777827721593], [93.1813888037532, 22.417778277215934], [93.18166526801849, 22.41805474148125]]}


In [7]:
def get_first_feature_properties(filepath):
        with open(filepath, 'r') as f:
            for item in ijson.items(f, 'features.item.properties', use_float=True):
                return item
        return {}


In [8]:
feature_properties = get_first_feature_properties(geojson_filepath)
print(feature_properties)

{'ORDER': 1, 'cat': 99069, 'end_node': '93.18166666666666, 22.418055555555554', 'id': 67750, 'network': 443, 'start_node': '93.18083333333333, 22.41777777777778', 'stream_typ': 'start', 'type_code': 0}
